# EDA (Encoded) — UCI Cannabis Risk Dataset

This notebook performs exploratory analysis on the encoded train split, examining feature-target relationships, class balance, and feature importance using mutual information — after encoding the cannabis risk binary target.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.metrics import mutual_info_score

In [ ]:
train_df = pd.read_csv('uci_cannabis_train.csv')

print(train_df.shape)
display(train_df.head())
print(train_df.columns.tolist())
print(train_df.isnull().sum())
print(train_df.describe())

In [ ]:
# Visualise class distribution
sns.countplot(x='cannabis_risk', data=train_df, palette=['#3B82F6','#EF4444'])
plt.xticks([0, 1], ['Non-user (0)', 'At Risk (1)'])
plt.title('cannabis_risk Distribution (Train Set)')
plt.xlabel('cannabis_risk')
plt.ylabel('Count')
plt.show()

print(train_df['cannabis_risk'].value_counts())

In [ ]:
# Missing values heatmap
sns.heatmap(train_df.isnull(), cbar=False, yticklabels=False)
plt.title('Missing Values Heatmap (Train Set)')
plt.show()

In [ ]:
# Feature distributions split by risk class
FEATURES = ['Age','Gender','Education','Country','Ethnicity',
            'Nscore','Escore','Oscore','Ascore','Cscore','Impulsive','SS']
LABELS   = ['Age','Gender','Education','Country','Ethnicity',
            'Neuroticism','Extraversion','Openness',
            'Agreeableness','Conscientiousness','Impulsivity','Sensation-Seeking']

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
axes = axes.flatten()
for i, (col, label) in enumerate(zip(FEATURES, LABELS)):
    axes[i].hist(train_df[train_df['cannabis_risk']==0][col],
                 bins=15, alpha=0.6, label='Non-user', color='#3B82F6', density=True)
    axes[i].hist(train_df[train_df['cannabis_risk']==1][col],
                 bins=15, alpha=0.6, label='At Risk',  color='#EF4444', density=True)
    axes[i].set_title(label, fontsize=9)
    axes[i].legend(fontsize=7)
fig.suptitle('Feature Distributions by Cannabis Risk Class', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Mutual information of each feature with cannabis_risk
# Note: features are continuous so we use a discretised approximation
# Higher MI = stronger statistical association with the target

mi_scores = {}
for col in FEATURES:
    # Discretise continuous features into 10 bins for MI calculation
    binned = pd.cut(train_df[col], bins=10, labels=False)
    mi_scores[col] = mutual_info_score(binned, train_df['cannabis_risk'])

mi_series = pd.Series(mi_scores, index=LABELS).sort_values()
mi_series.plot(kind='barh', figsize=(9, 6), color='steelblue', alpha=0.85)
plt.xlabel('Mutual Information Score')
plt.title('Feature-Target Mutual Information (cannabis_risk)')
plt.tight_layout()
plt.show()

print('\nMutual Information Scores:')
for label, score in zip(LABELS, [mi_scores[f] for f in FEATURES]):
    print(f'  {label:<22}: {score:.4f}')

- All 12 features show **non-zero mutual information** with the cannabis risk target — confirming that these features carry genuine predictive signal.
- **Sensation-Seeking** and **Country** show the highest association with cannabis risk.
- **Openness**, **Conscientiousness**, and **Age** also show meaningful MI scores.
- This is in sharp contrast to the previous dataset where all features returned MI = 0.0000.

In [ ]:
# Correlation heatmap of features in encoded train set
corr = train_df[FEATURES].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            xticklabels=LABELS, yticklabels=LABELS)
plt.title('Feature Correlation Matrix (Encoded Train Set)')
plt.tight_layout()
plt.show()

In [ ]:
# Mean feature values by cannabis risk class
mean_by_class = train_df.groupby('cannabis_risk')[FEATURES].mean()
mean_by_class.index = ['Non-user (0)', 'At Risk (1)']
mean_by_class.columns = LABELS
mean_by_class.T.plot(kind='bar', figsize=(12, 5), color=['#3B82F6','#EF4444'], alpha=0.85)
plt.title('Mean Feature Values by Cannabis Risk Class')
plt.ylabel('Mean Quantified Value')
plt.xticks(rotation=30, ha='right')
plt.legend()
plt.tight_layout()
plt.show()

display(mean_by_class.T)